In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4


def generate_id() -> str:
    """Generate a unique application ID."""
    return str(uuid4())


def utc_now() -> datetime:
    """Return the current timezone-aware UTC datetime."""
    return datetime.now(timezone.utc)


def ensure_directory(path: str | Path) -> Path:
    """Create a directory if it does not exist."""
    directory = Path(path)
    directory.mkdir(parents=True, exist_ok=True)
    return directory


def safe_filename(filename: str) -> str:
    """
    Convert a user-provided filename into a safer filename.

    This prevents path traversal through the filename itself.
    """
    filename = Path(filename).name

    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        "._-"
    )

    cleaned = "".join(
        character if character in allowed else "_"
        for character in filename
    )

    return cleaned or "uploaded_file"

def record_activity(
    database,
    *,
    user_id: str,
    project_id: str | None,
    event_type: str,
    description: str | None = None,
    entity_type: str | None = None,
    entity_id: str | None = None,
    metadata: dict | None = None,
):
    document = {
        "id": generate_id(),
        "user_id": user_id,
        "project_id": project_id,
        "event_type": event_type,
        "description": description,
        "entity_type": entity_type,
        "entity_id": entity_id,
        "metadata": metadata or {},
        "created_at": utc_now(),
    }

    try:
        database.collection("activities").insert_one(document)
    except Exception as exc:
        import logging
        logging.getLogger(__name__).warning(
            "Failed to record activity: %s",
            exc,
        )

    return document

def clamp(
    value: float,
    minimum: float = 0.0,
    maximum: float = 1.0,
) -> float:
    """Keep a numeric value inside a specified range."""
    return max(minimum, min(value, maximum))
